In [11]:
using LinearAlgebra
using Random
using Printf
using BenchmarkTools

# ============================================================
# Matrix-free 2D Poisson operator
#
# A = (I ⊗ T + T ⊗ I) / h^2
#
# where T = tridiag(-1, 2, -1)
# ============================================================

function Amul!(y, x, N, h2inv)
    X = reshape(x, N, N)
    Y = reshape(y, N, N)

    @inbounds for j in 1:N
        for i in 1:N
            v = 4 * X[i, j]

            i > 1 && (v -= X[i-1, j])
            i < N && (v -= X[i+1, j])

            j > 1 && (v -= X[i, j-1])
            j < N && (v -= X[i, j+1])

            Y[i, j] = h2inv * v
        end
    end

    return y
end


# ============================================================
# Majorizer
#
# B = I ⊗ M
#
# M = h^{-2} tridiag(-1, α, -1)
#
# α = 4 + 2 cos(pi/(N+1))
#
# Write
#
# M = h^{-2} L L'
#
# where L is lower bidiagonal.
#
# Store only:
#
#   ell[i] = diagonal of L
#   c[i]   = L[i+1,i]
#
# No M or B needs to be explicitly stored.
# ============================================================

function build_majorizer(N)
    α = 4 + 2cos(pi / (N + 1))

    ell = Vector{Float64}(undef, N)
    c   = Vector{Float64}(undef, N - 1)

    ell[1] = sqrt(α)

    @inbounds for i in 2:N
        c[i-1] = -1 / ell[i-1]
        ell[i] = sqrt(α - c[i-1]^2)
    end

    return ell, c
end


# ============================================================
# B^{-1} r: column-oriented implementation
#
# Solve
#
#   M Z = R
#
# where
#
#   r = vec(R)
#   z = vec(Z)
#
# Since
#
#   M = h^{-2} L L',
#
# solve
#
#   L Y  = h^2 R
#   L' Z = Y
#
# This version follows Julia's column-major memory layout.
# ============================================================

function Bsolve_col!(z, r, ell, c, N, h2)
    R = reshape(r, N, N)
    Z = reshape(z, N, N)

    # Forward solve:
    #
    # L Y = h^2 R

    @inbounds for j in 1:N
        Z[1, j] = h2 * R[1, j] / ell[1]

        for i in 2:N
            Z[i, j] =
                (h2 * R[i, j] -
                 c[i-1] * Z[i-1, j]) / ell[i]
        end
    end

    # Backward solve:
    #
    # L' Z = Y

    @inbounds for j in 1:N
        Z[N, j] /= ell[N]

        for i in N-1:-1:1
            Z[i, j] =
                (Z[i, j] -
                 c[i] * Z[i+1, j]) / ell[i]
        end
    end

    return z
end


# ============================================================
# B^{-1} r: SIMD across right-hand sides
#
# Mathematically identical to Bsolve_col!.
#
# Here i is the recurrence direction, while different RHS
# indexed by j are independent.
#
# Whether this is faster depends on memory layout / compiler.
# ============================================================

function Bsolve_simd!(z, r, ell, c, N, h2)
    R = reshape(r, N, N)
    Z = reshape(z, N, N)

    # --------------------------------------------------------
    # Forward solve
    # --------------------------------------------------------

    ell1 = ell[1]

    @inbounds @simd for j in 1:N
        Z[1, j] = h2 * R[1, j] / ell1
    end

    @inbounds for i in 2:N
        ci = c[i-1]
        li = ell[i]

        @simd for j in 1:N
            Z[i, j] =
                (h2 * R[i, j] -
                 ci * Z[i-1, j]) / li
        end
    end

    # --------------------------------------------------------
    # Backward solve
    # --------------------------------------------------------

    ellN = ell[N]

    @inbounds @simd for j in 1:N
        Z[N, j] /= ellN
    end

    @inbounds for i in N-1:-1:1
        ci = c[i]
        li = ell[i]

        @simd for j in 1:N
            Z[i, j] =
                (Z[i, j] -
                 ci * Z[i+1, j]) / li
        end
    end

    return z
end


# ============================================================
# MM
#
# u_{k+1}
# =
# u_k + B^{-1}(b - A u_k)
# ============================================================

function poisson_mm(
    b,
    N,
    h,
    ell,
    c;
    tol=1e-8,
    maxiter=100_000,
    Bsolve! = Bsolve_col!
)

    T = eltype(b)

    u  = zeros(T, length(b))
    Au = similar(b)
    r  = similar(b)
    d  = similar(b)

    bn = norm(b)

    h2    = h^2
    h2inv = inv(h2)

    for k in 0:maxiter

        Amul!(Au, u, N, h2inv)

        @. r = b - Au

        rr = dot(r, r)
        relres = sqrt(rr) / bn

        if relres <= tol
            return u, k, relres, true
        end

        k == maxiter && break

        Bsolve!(d, r, ell, c, N, h2)

        @. u += d
    end

    relres = norm(r) / bn

    return u, maxiter, relres, false
end


# ============================================================
# CG
# ============================================================

function poisson_cg(
    b,
    N,
    h;
    tol=1e-8,
    maxiter=10_000
)

    T = eltype(b)

    u  = zeros(T, length(b))
    r  = copy(b)
    p  = copy(b)
    Ap = similar(b)

    bn = norm(b)
    rr = dot(r, r)

    h2inv = inv(h^2)

    for k in 1:maxiter

        Amul!(Ap, p, N, h2inv)

        α = rr / dot(p, Ap)

        @. u += α * p
        @. r -= α * Ap

        rrnew = dot(r, r)

        relres = sqrt(rrnew) / bn

        if relres <= tol
            return u, k, relres, true
        end

        β = rrnew / rr

        @. p = r + β * p

        rr = rrnew
    end

    relres = norm(r) / bn

    return u, maxiter, relres, false
end


# ============================================================
# PCG
#
# Preconditioner:
#
# P = B
#
# z = B^{-1} r
# ============================================================

function poisson_pcg(
    b,
    N,
    h,
    ell,
    c;
    tol=1e-8,
    maxiter=10_000,
    Bsolve! = Bsolve_col!
)

    T = eltype(b)

    u  = zeros(T, length(b))
    r  = copy(b)
    z  = similar(b)
    p  = similar(b)
    Ap = similar(b)

    bn = norm(b)

    h2    = h^2
    h2inv = inv(h2)

    # Initial preconditioning
    Bsolve!(z, r, ell, c, N, h2)

    copyto!(p, z)

    rz = dot(r, z)

    for k in 1:maxiter

        # A p
        Amul!(Ap, p, N, h2inv)

        α = rz / dot(p, Ap)

        @. u += α * p
        @. r -= α * Ap

        rr = dot(r, r)
        relres = sqrt(rr) / bn

        if relres <= tol
            return u, k, relres, true
        end

        # z = B^{-1} r
        Bsolve!(z, r, ell, c, N, h2)

        rznew = dot(r, z)

        β = rznew / rz

        @. p = z + β * p

        rz = rznew
    end

    relres = norm(r) / bn

    return u, maxiter, relres, false
end


# ============================================================
# Verify Bsolve!
#
# Compare against Julia's SymTridiagonal factorization.
# ============================================================

function check_Bsolve(N, h, ell, c; Bsolve! = Bsolve_col!)
    α = 4 + 2cos(pi / (N + 1))

    M = SymTridiagonal(
        fill(α / h^2, N),
        fill(-1 / h^2, N - 1)
    )

    F = cholesky(M)

    r = randn(N^2)

    # Structured solver
    z1 = similar(r)

    Bsolve!(
        z1,
        r,
        ell,
        c,
        N,
        h^2
    )

    # Reference solver
    R = reshape(r, N, N)
    Z = copy(R)

    ldiv!(F, Z)

    z2 = vec(Z)

    return norm(z1 - z2) / norm(z2)
end


# ============================================================
# Experiment
# ============================================================

Random.seed!(1234)

N   = 100
h   = 1 / (N + 1)
tol = 1e-8

h2inv = inv(h^2)

# ------------------------------------------------------------
# Generate exact solution
# ------------------------------------------------------------

utrue = randn(N^2)
utrue ./= norm(utrue)

b = similar(utrue)

Amul!(
    b,
    utrue,
    N,
    h2inv
)

# ------------------------------------------------------------
# Build reusable majorizer factors
# ------------------------------------------------------------

ell, c = build_majorizer(N)


# ============================================================
# Check structured solvers
# ============================================================

err_col =
    check_Bsolve(
        N,
        h,
        ell,
        c;
        Bsolve! = Bsolve_col!
    )

err_simd =
    check_Bsolve(
        N,
        h,
        ell,
        c;
        Bsolve! = Bsolve_simd!
    )

println("Bsolve verification:")
@printf("column version = %.3e\n", err_col)
@printf("SIMD version   = %.3e\n", err_simd)
println()


# ============================================================
# Solve
#
# Start with column version.
# Change Bsolve! = Bsolve_simd! below if SIMD wins benchmark.
# ============================================================

umm, imm, rmm, cmm =
    poisson_mm(
        b,
        N,
        h,
        ell,
        c;
        tol=tol,
        Bsolve! = Bsolve_col!
    )

ucg, icg, rcg, ccg =
    poisson_cg(
        b,
        N,
        h;
        tol=tol
    )

upcg, ipcg, rpcg, cpcg =
    poisson_pcg(
        b,
        N,
        h,
        ell,
        c;
        tol=tol,
        Bsolve! = Bsolve_col!
    )


# ============================================================
# Results
# ============================================================

println("N        = ", N)
println("Unknowns = ", N^2)
println()

@printf(
    "%-12s %8s %14s %12s %14s\n",
    "Method",
    "Iter",
    "RelRes",
    "Converged",
    "RelError"
)

@printf(
    "%-12s %8d %14.3e %12s %14.3e\n",
    "MM",
    imm,
    rmm,
    string(cmm),
    norm(umm - utrue) / norm(utrue)
)

@printf(
    "%-12s %8d %14.3e %12s %14.3e\n",
    "CG",
    icg,
    rcg,
    string(ccg),
    norm(ucg - utrue) / norm(utrue)
)

@printf(
    "%-12s %8d %14.3e %12s %14.3e\n",
    "PCG",
    ipcg,
    rpcg,
    string(cpcg),
    norm(upcg - utrue) / norm(utrue)
)


# ============================================================
# Microbenchmarks
# ============================================================

println("\n============================================================")
println("Kernel benchmarks")
println("============================================================")

Au = similar(b)
z  = similar(b)
r  = copy(b)

bench_A = @benchmark Amul!(
    $Au,
    $b,
    $N,
    $h2inv
)

bench_B_col = @benchmark Bsolve_col!(
    $z,
    $r,
    $ell,
    $c,
    $N,
    $(h^2)
)

bench_B_simd = @benchmark Bsolve_simd!(
    $z,
    $r,
    $ell,
    $c,
    $N,
    $(h^2)
)

@printf(
    "A*x              = %.3f μs\n",
    median(bench_A).time / 1e3
)

@printf(
    "B^-1*r (column)  = %.3f μs\n",
    median(bench_B_col).time / 1e3
)

@printf(
    "B^-1*r (SIMD)    = %.3f μs\n",
    median(bench_B_simd).time / 1e3
)


# ============================================================
# Full solver benchmarks
# ============================================================

println("\n============================================================")
println("Solver benchmarks")
println("============================================================")

bcg = @benchmark poisson_cg(
    $b,
    $N,
    $h;
    tol=$tol
)

bpcg_col = @benchmark poisson_pcg(
    $b,
    $N,
    $h,
    $ell,
    $c;
    tol=$tol,
    Bsolve! = Bsolve_col!
)

bpcg_simd = @benchmark poisson_pcg(
    $b,
    $N,
    $h,
    $ell,
    $c;
    tol=$tol,
    Bsolve! = Bsolve_simd!
)

println("\nMedian runtime:")

@printf(
    "CG          = %.6f s\n",
    median(bcg).time / 1e9
)

@printf(
    "PCG column  = %.6f s\n",
    median(bpcg_col).time / 1e9
)

@printf(
    "PCG SIMD    = %.6f s\n",
    median(bpcg_simd).time / 1e9
)

Bsolve verification:
column version = 2.010e-16
SIMD version   = 2.003e-16

N        = 100
Unknowns = 10000

Method           Iter         RelRes    Converged       RelError
MM              10595      9.999e-09         true      2.323e-05
CG                261      9.229e-09         true      3.614e-07
PCG               185      9.136e-09         true      3.547e-07

Kernel benchmarks
A*x              = 6.104 μs
B^-1*r (column)  = 43.416 μs
B^-1*r (SIMD)    = 9.278 μs

Solver benchmarks

Median runtime:
CG          = 0.003799 s
PCG column  = 0.010735 s
PCG SIMD    = 0.004640 s


In [13]:
function scaling_experiment(
    Ns;
    tol=1e-8,
    seed=1234
)
    println()
    println("="^100)
    println("SCALING EXPERIMENT")
    println("="^100)

    @printf(
        "%6s %12s %10s %10s %12s %12s %10s %10s\n",
        "N",
        "n",
        "CG iter",
        "PCG iter",
        "CG time",
        "PCG time",
        "IterRatio",
        "TimeRatio"
    )

    results = []

    for N in Ns
        Random.seed!(seed)

        n = N^2
        h = 1 / (N + 1)
        h2inv = inv(h^2)

        # ----------------------------------------------------
        # Generate problem
        # ----------------------------------------------------

        utrue = randn(n)
        utrue ./= norm(utrue)

        b = similar(utrue)

        Amul!(
            b,
            utrue,
            N,
            h2inv
        )

        # ----------------------------------------------------
        # Majorizer
        # ----------------------------------------------------

        ell, c = build_majorizer(N)

        # ----------------------------------------------------
        # Solve once: iteration counts
        # ----------------------------------------------------

        ucg, icg, rcg, ccg =
            poisson_cg(
                b,
                N,
                h;
                tol=tol
            )

        upcg, ipcg, rpcg, cpcg =
            poisson_pcg(
                b,
                N,
                h,
                ell,
                c;
                tol=tol,
                Bsolve! = Bsolve_simd!
            )

        # ----------------------------------------------------
        # Benchmark
        # ----------------------------------------------------

        tcg = @belapsed poisson_cg(
            $b,
            $N,
            $h;
            tol=$tol
        )

        tpcg = @belapsed poisson_pcg(
            $b,
            $N,
            $h,
            $ell,
            $c;
            tol=$tol,
            Bsolve! = Bsolve_simd!
        )

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        iter_ratio = ipcg / icg
        time_ratio = tpcg / tcg

        err_cg =
            norm(ucg - utrue) / norm(utrue)

        err_pcg =
            norm(upcg - utrue) / norm(utrue)

        push!(
            results,
            (
                N=N,
                n=n,

                cg_iter=icg,
                pcg_iter=ipcg,

                cg_relres=rcg,
                pcg_relres=rpcg,

                cg_error=err_cg,
                pcg_error=err_pcg,

                cg_time=tcg,
                pcg_time=tpcg,

                iter_ratio=iter_ratio,
                time_ratio=time_ratio
            )
        )

        @printf(
            "%6d %12d %10d %10d %12.6f %12.6f %10.3f %10.3f\n",
            N,
            n,
            icg,
            ipcg,
            tcg,
            tpcg,
            iter_ratio,
            time_ratio
        )
    end

    return results
end


# ============================================================
# Run scaling experiment
# ============================================================

Ns = [50, 100, 200, 500, 1000]

results = scaling_experiment(
    Ns;
    tol=1e-8
)


SCALING EXPERIMENT
     N            n    CG iter   PCG iter      CG time     PCG time  IterRatio  TimeRatio
    50         2500        139         98     0.000380     0.000600      0.705      1.577
   100        10000        261        185     0.003505     0.004314      0.709      1.231
   200        40000        474        336     0.033188     0.040871      0.709      1.231
   500       250000        990        701     0.376427     0.449582      0.708      1.194
  1000      1000000       1265        894     1.878454     2.403847      0.707      1.280


5-element Vector{Any}:
 (N = 50, n = 2500, cg_iter = 139, pcg_iter = 98, cg_relres = 8.248170309536074e-9, pcg_relres = 9.744032866589637e-9, cg_error = 9.747306139706416e-8, pcg_error = 2.4309219394057445e-7, cg_time = 0.000380333, pcg_time = 0.000599833, iter_ratio = 0.7050359712230215, time_ratio = 1.5771258344661125)
 (N = 100, n = 10000, cg_iter = 261, pcg_iter = 185, cg_relres = 9.229402504427761e-9, pcg_relres = 9.136095181620436e-9, cg_error = 3.6141283118220246e-7, pcg_error = 3.5470348913800117e-7, cg_time = 0.003504708, pcg_time = 0.004314333, iter_ratio = 0.7088122605363985, time_ratio = 1.231010686196967)
 (N = 200, n = 40000, cg_iter = 474, pcg_iter = 336, cg_relres = 9.911848036296933e-9, pcg_relres = 9.38692363159436e-9, cg_error = 1.2604991281886834e-6, pcg_error = 1.199233331943942e-6, cg_time = 0.033187917, pcg_time = 0.040870667, iter_ratio = 0.7088607594936709, time_ratio = 1.231492383206816)
 (N = 500, n = 250000, cg_iter = 990, pcg_iter = 701, cg_relres = 9.92654